In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os   
import json

load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")
""" base_url = os.getenv("DASHSCOPE_API_URL")  """
base_url_response = os.getenv("DASHSCOPE_API_URL_RESPONSE")

client = OpenAI(
    api_key=api_key,
    base_url=base_url_response
)

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_horoscope",
        "description": "Get today's horoscope for an astrological sign.",
        "parameters": {
            "type": "object",
            "properties": {
                "sign": {
                    "type": "string",
                    "description": "An astrological sign like Taurus or Aquarius",
                },
            },
            "required": ["sign"],
        },
    },
]


def get_horoscope(sign):
    return f"{sign}: Next Tuesday you will befriend a baby otter."


# Create a running input list we will add to over time
input_list = [{"role": "user", "content": "What is my horoscope? I am an Aquarius."}]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model="qwen3.8-max",
    tools=tools,
    input=input_list,
)

# Save function call outputs for subsequent requests
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_horoscope":
            # 3. Execute the function logic for get_horoscope
            sign = json.loads(item.arguments)["sign"]
            horoscope = get_horoscope(sign)

            # 4. Provide function call results to the model
            input_list.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": horoscope,
                }
            )

print("Final input:")
print(input_list)

response = client.responses.create(
    model="qwen3.8-max",
    instructions="Respond only with a horoscope generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os   
import json

load_dotenv()

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url_response = os.getenv("DASHSCOPE_API_URL_RESPONSE")

client = OpenAI(
    api_key=api_key,
    base_url=base_url_response
)

tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "查询当前城市的天气",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "要查询的城市的名称 如 北京 上海 西安等"
                },
                "units": {
                    "type": "string",
                    "enum": ["摄氏度", "华氏度"],
                    "description": "要返回的温度单位"
                }
            },
            "required": ["city"]
        },  
    }, 
]

def get_weather(city):
    return f"当前{city}的天气是晴朗的,温度为35摄氏度"

input_list =[{"role": "user", "content": "西安天气如何"}]

response = client.responses.create(
    model="qwen3.8-max",
    tools=tools,
    input=input_list,
)

print(response.output)

input_list += response.output

for chunk in response.output:
    if chunk.type == "function_call":
        if chunk.name == "get_weather":
              city = json.loads(chunk.arguments)["city"]
              weather = get_weather(city)
              
              input_list.append(
                {
                    "type": "function_call_output",
                    "call_id": chunk.call_id,
                    "output": weather
                }
              )

response = client.responses.create(
    model="qwen3.8-max",
    instructions="请根据用户的问题调用函数来获取天气信息",
    tools=tools,
    input=input_list,
)

print(response.model_dump_json(indent=2))
print("\n" + response.output_text)


tools[
    {
        type: "function",
        name: "函数名"
        description: "函数描述"
        strict: true, # 严格模式下，参数必须包含在参数列表中
        parameters: "函数参数"
        {
            type: "object",
            properties: {
                city(参数名): {
                    type: "参数类型",
                    description: "参数描述"
                },
                units(参数名): {
                    type: "参数类型",
                    description: "参数描述"
                }
            }
            required: ["city", "units"]
        }
    },
    {
        工具2
    }
]